Derm7pt -- loss ablations on the best config (efficientnet_b0 @ 512px + meta-data)

Before running:

Attach the same shared derm7pt-release-v0 dataset.
Notebook settings -> Accelerator: GPU T4 x2. Internet: On.
Three loss-ablation configs on the overall-best backbone/resolution (efficientnet_b0 @ 512px+meta, test_diagnosis_5way_acc=0.6734): (a) baseline rerun (learned uncertainty weighting + class-weighted CE), (b) fixed clinical-checklist task weights, (c) focal loss for the diagnosis task. Baseline is rerun (not reused from the earlier notebook) so per-class F1 is directly comparable across all three.

In [ ]:
import os, time, csv, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score

try:
    import timm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))

torch 2.11.0+cu130 | cuda available: True | GPUs: 2
 - Tesla T4
 - Tesla T4


In [ ]:
# ---- CONFIG ----
import glob

def _autodetect_input_dir():
    hits = glob.glob('/kaggle/input/**/meta/meta.csv', recursive=True)
    if hits:
        return os.path.dirname(os.path.dirname(hits[0]))
    return None

_detected = _autodetect_input_dir()
if _detected:
    INPUT_DIR = _detected
    print('Auto-detected INPUT_DIR:', INPUT_DIR)
else:
    _fallback_candidates = [
        '/kaggle/input/datasets/noorabdullah02/derm7pt-release-v0/release_v0',
        '/kaggle/input/derm7pt-release-v0/release_v0',
        '/kaggle/input/derm7pt/release_v0',
    ]
    INPUT_DIR = next((p for p in _fallback_candidates if os.path.isdir(p)), _fallback_candidates[0])
    print('Could not auto-detect dataset path. Falling back to', INPUT_DIR, '-- edit manually if wrong.')

OUTPUT_DIR = '/kaggle/working'
IMG_SIZE = 512
BACKBONE = 'efficientnet_b0'
BATCH_SIZE = 8
EPOCHS = 40
EARLY_STOP_PATIENCE = 12
LR = 2e-4
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MULTI_GPU = torch.cuda.device_count() > 1
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

if not os.path.isdir(INPUT_DIR):
    zip_candidates = [p for p in os.listdir('/kaggle/input') if p.endswith('.zip')]
    print('INPUT_DIR not found as a folder. Zip files seen in /kaggle/input:', zip_candidates)

Auto-detected INPUT_DIR: /kaggle/input/datasets/noorabdullah02/derm7pt-release-v0/release_v0


In [ ]:
# ---- the 3 loss-ablation configs (same backbone/resolution/batch-size throughout) ----
ABLATION_CONFIGS = {
    'baseline':  {'loss_mode': 'uncertainty', 'diagnosis_loss': 'ce'},
    'clinical':  {'loss_mode': 'clinical',     'diagnosis_loss': 'ce'},
    'focal':     {'loss_mode': 'uncertainty', 'diagnosis_loss': 'focal'},
}

In [ ]:
# ---- write standalone module files for the isolated-subprocess trainer ----
_MODULE_FILES = {
    "labels.py": '"""Label taxonomy for the Derm7pt dataset: diagnosis grouping + 7-point criteria encodings."""\n\n# Melanoma subtypes collapse into a single "melanoma" class (standard grouping,\n# matches the original JBHI 2019 paper\'s 15-way grouped diagnosis task).\nDIAGNOSIS_GROUPS = {\n    "basal cell carcinoma": "basal cell carcinoma",\n    "blue nevus": "blue nevus",\n    "clark nevus": "clark nevus",\n    "combined nevus": "combined nevus",\n    "congenital nevus": "congenital nevus",\n    "dermal nevus": "dermal nevus",\n    "dermatofibroma": "dermatofibroma",\n    "lentigo": "lentigo",\n    "melanoma": "melanoma",\n    "melanoma (in situ)": "melanoma",\n    "melanoma (less than 0.76 mm)": "melanoma",\n    "melanoma (0.76 to 1.5 mm)": "melanoma",\n    "melanoma (more than 1.5 mm)": "melanoma",\n    "melanoma metastasis": "melanoma",\n    "melanosis": "melanosis",\n    "miscellaneous": "miscellaneous",\n    "recurrent nevus": "recurrent nevus",\n    "reed or spitz nevus": "reed or spitz nevus",\n    "seborrheic keratosis": "seborrheic keratosis",\n    "vascular lesion": "vascular lesion",\n}\nDIAGNOSIS_CLASSES = sorted(set(DIAGNOSIS_GROUPS.values()))\n\n# Binary malignancy grouping, useful for a benign-vs-malignant head.\nMALIGNANT_DIAGNOSES = {"melanoma", "basal cell carcinoma"}\n\n# Coarser 5-way diagnosis grouping (BCC, NEV, MEL, MISC, SK) matching the\n# original Kawahara et al. 2019 JBHI paper\'s Table I/IV, so our finer-grained\n# 15-way predictions can be regrouped post-hoc for a direct accuracy\n# comparison against the paper\'s reported baseline (74.2% on the test set).\nFIVE_WAY_GROUPS = {\n    "basal cell carcinoma": "BCC",\n    "blue nevus": "NEV",\n    "clark nevus": "NEV",\n    "combined nevus": "NEV",\n    "congenital nevus": "NEV",\n    "dermal nevus": "NEV",\n    "recurrent nevus": "NEV",\n    "reed or spitz nevus": "NEV",\n    "melanoma": "MEL",\n    "dermatofibroma": "MISC",\n    "lentigo": "MISC",\n    "melanosis": "MISC",\n    "miscellaneous": "MISC",\n    "vascular lesion": "MISC",\n    "seborrheic keratosis": "SK",\n}\nFIVE_WAY_CLASSES = sorted(set(FIVE_WAY_GROUPS.values()))\n\n# The seven 7-point-checklist criteria columns in meta.csv, with their label vocab.\nSEVEN_POINT_CRITERIA = {\n    "pigment_network": ["absent", "typical", "atypical"],\n    "blue_whitish_veil": ["absent", "present"],\n    "vascular_structures": [\n        "absent", "arborizing", "comma", "hairpin",\n        "within regression", "wreath", "dotted", "linear irregular",\n    ],\n    "pigmentation": [\n        "absent", "diffuse regular", "localized regular",\n        "diffuse irregular", "localized irregular",\n    ],\n    "streaks": ["absent", "regular", "irregular"],\n    "dots_and_globules": ["absent", "regular", "irregular"],\n    "regression_structures": ["absent", "blue areas", "white areas", "combinations"],\n}\n\n\ndef diagnosis_to_label(diagnosis: str) -> str:\n    return DIAGNOSIS_GROUPS[diagnosis.strip().lower()]\n\n\ndef diagnosis_to_malignant(diagnosis: str) -> int:\n    return int(diagnosis_to_label(diagnosis) in MALIGNANT_DIAGNOSES)\n\n\ndef criterion_to_index(column: str, value: str) -> int:\n    vocab = SEVEN_POINT_CRITERIA[column]\n    return vocab.index(value.strip().lower())\n\n\n# Patient meta-data used as an auxiliary model input, matching the original\n# paper\'s "patient meta-data" (gender, lesion location, lesion elevation).\nMETA_FIELDS = {\n    "sex": ["female", "male"],\n    "location": [\n        "abdomen", "acral", "back", "buttocks", "chest",\n        "genital areas", "head neck", "lower limbs", "upper limbs",\n    ],\n    "elevation": ["flat", "nodular", "palpable"],\n}\nMETA_DIM = sum(len(vocab) for vocab in META_FIELDS.values())\n\n\ndef encode_meta(row) -> list:\n    """One-hot encodes the sex/location/elevation meta-data fields into a flat\n    vector. An unrecognized or missing value falls back to all-zeros for that\n    field rather than raising, since meta-data is auxiliary, not a label.\n    """\n    vec = []\n    for field, vocab in META_FIELDS.items():\n        value = str(row[field]).strip().lower()\n        vec.extend(1.0 if value == v else 0.0 for v in vocab)\n    return vec\n',
    "dataset.py": '"""PyTorch Dataset for Derm7pt: loads clinical+dermoscopy image pairs with\ndiagnosis and 7-point-checklist criteria labels, using the official data splits.\n"""\nimport os\n\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom labels import (\n    DIAGNOSIS_CLASSES,\n    SEVEN_POINT_CRITERIA,\n    criterion_to_index,\n    diagnosis_to_label,\n    diagnosis_to_malignant,\n    encode_meta,\n)\n\nCRITERIA_COLUMNS = list(SEVEN_POINT_CRITERIA.keys())\n\n\nclass Derm7ptDataset(Dataset):\n    """One sample = one lesion case: a (clinical, dermoscopy) image pair,\n    the grouped diagnosis label, a binary malignancy label, and the 7\n    criteria labels (for multi-task training).\n    """\n\n    def __init__(self, release_dir: str, split: str, transform=None):\n        """\n        Args:\n            release_dir: path to .../derm7pt/release_v0\n            split: one of "train", "valid", "test"\n            transform: torchvision transform applied to both clinic & derm images\n        """\n        assert split in ("train", "valid", "test")\n        meta_dir = os.path.join(release_dir, "meta")\n        self.images_dir = os.path.join(release_dir, "images")\n        self.transform = transform\n\n        meta = pd.read_csv(os.path.join(meta_dir, "meta.csv"))\n        split_indexes = pd.read_csv(os.path.join(meta_dir, f"{split}_indexes.csv"))["indexes"]\n        self.df = meta.iloc[split_indexes].reset_index(drop=True)\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n\n        clinic_img = Image.open(os.path.join(self.images_dir, row["clinic"])).convert("RGB")\n        derm_img = Image.open(os.path.join(self.images_dir, row["derm"])).convert("RGB")\n        if self.transform is not None:\n            clinic_img = self.transform(clinic_img)\n            derm_img = self.transform(derm_img)\n\n        diag_label = DIAGNOSIS_CLASSES.index(diagnosis_to_label(row["diagnosis"]))\n        malignant_label = diagnosis_to_malignant(row["diagnosis"])\n        criteria_labels = {\n            col: criterion_to_index(col, row[col]) for col in CRITERIA_COLUMNS\n        }\n\n        return {\n            "clinic": clinic_img,\n            "derm": derm_img,\n            "meta": torch.tensor(encode_meta(row), dtype=torch.float32),\n            "diagnosis": torch.tensor(diag_label, dtype=torch.long),\n            "malignant": torch.tensor(malignant_label, dtype=torch.long),\n            **{\n                col: torch.tensor(val, dtype=torch.long)\n                for col, val in criteria_labels.items()\n            },\n            "case_num": int(row["case_num"]),\n        }\n\n\nif __name__ == "__main__":\n    import sys\n\n    release_dir = sys.argv[1] if len(sys.argv) > 1 else r"C:\\Users\\user\\Desktop\\DermPt7t\\derm7pt\\release_v0"\n    for split in ("train", "valid", "test"):\n        ds = Derm7ptDataset(release_dir, split)\n        print(f"{split}: {len(ds)} cases")\n        sample = ds[0]\n        print(\n            "  sample keys:", list(sample.keys()),\n            "| clinic size:", sample["clinic"].size,\n            "| diagnosis idx:", sample["diagnosis"].item(),\n        )\n',
    "model.py": '"""Multi-task, multi-modal (clinic + dermoscopy) model for Derm7pt.\n\nOne shared-weight image backbone (any timm model) encodes both the clinical\nand dermoscopic photo; the two feature vectors are concatenated and fed into\nper-task heads: diagnosis (15-way), malignant (binary), and the 7 checklist\ncriteria (each its own small classification head).\n"""\nimport timm\nimport torch\nimport torch.nn as nn\n\nfrom labels import DIAGNOSIS_CLASSES, META_DIM, SEVEN_POINT_CRITERIA\n\nCRITERIA_NUM_CLASSES = {k: len(v) for k, v in SEVEN_POINT_CRITERIA.items()}\n\n# Fixed task weights mirroring the real clinical 7-point checklist\'s point\n# values (pigment network / blue-whitish-veil / vascular structures = 2 pts,\n# the other four criteria = 1 pt each). diagnosis and malignant are weighted\n# on par with the top-value criteria since they\'re the primary clinical\n# outcomes. Used by `multitask_loss_clinical` as an ablation against the\n# default learned uncertainty weighting.\nCLINICAL_TASK_WEIGHTS = {\n    "diagnosis": 2.0,\n    "malignant": 2.0,\n    "pigment_network": 2.0,\n    "blue_whitish_veil": 2.0,\n    "vascular_structures": 2.0,\n    "pigmentation": 1.0,\n    "streaks": 1.0,\n    "dots_and_globules": 1.0,\n    "regression_structures": 1.0,\n}\n\n\nclass MultiTaskDerm7pt(nn.Module):\n    def __init__(self, backbone_name: str, pretrained: bool = True, dropout: float = 0.3,\n                 meta_dim: int = META_DIM, img_size: int = 224):\n        super().__init__()\n        try:\n            # window/grid-based archs (e.g. swin) bake the resolution into their\n            # layout and need it passed explicitly to run at anything but 224px\n            self.backbone = timm.create_model(\n                backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg",\n                img_size=img_size,\n            )\n        except TypeError:\n            # most conv backbones (effnet, densenet, ...) use global pooling and\n            # don\'t accept/need an img_size kwarg at all\n            self.backbone = timm.create_model(\n                backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg"\n            )\n        # `backbone.num_features` is unreliable for some archs (e.g. MobileNetV3\'s\n        # head-expansion conv isn\'t reflected in it) — probe the real output dim instead.\n        with torch.no_grad():\n            feat_dim = self.backbone(torch.zeros(1, 3, img_size, img_size)).shape[1]\n        # clinic + derm visual features, plus one-hot patient meta-data (sex,\n        # lesion location, elevation) -- matches the original paper\'s input modalities\n        fused_dim = feat_dim * 2 + meta_dim\n\n        self.dropout = nn.Dropout(dropout)\n        self.diagnosis_head = nn.Linear(fused_dim, len(DIAGNOSIS_CLASSES))\n        self.malignant_head = nn.Linear(fused_dim, 2)\n        self.criteria_heads = nn.ModuleDict(\n            {name: nn.Linear(fused_dim, n) for name, n in CRITERIA_NUM_CLASSES.items()}\n        )\n\n        # Homoscedastic uncertainty weighting (Kendall et al., 2018): one learnable\n        # log-variance per task, so easy tasks don\'t drown out hard ones during\n        # multi-task training, and the learned variances double as calibration signal.\n        task_names = ["diagnosis", "malignant"] + list(CRITERIA_NUM_CLASSES.keys())\n        self.log_vars = nn.ParameterDict(\n            {name: nn.Parameter(torch.zeros(())) for name in task_names}\n        )\n\n    def forward(self, clinic: torch.Tensor, derm: torch.Tensor, meta: torch.Tensor) -> dict:\n        feat_clinic = self.backbone(clinic)\n        feat_derm = self.backbone(derm)\n        fused = self.dropout(torch.cat([feat_clinic, feat_derm, meta], dim=1))\n\n        out = {\n            "diagnosis": self.diagnosis_head(fused),\n            "malignant": self.malignant_head(fused),\n        }\n        for name, head in self.criteria_heads.items():\n            out[name] = head(fused)\n        return out\n\n    def multitask_loss(\n        self,\n        outputs: dict,\n        targets: dict,\n        label_smoothing: float = 0.1,\n        diagnosis_class_weights: torch.Tensor | None = None,\n        diagnosis_loss_fn=None,\n    ) -> tuple[torch.Tensor, dict]:\n        """Uncertainty-weighted sum of per-task cross-entropy losses.\n        loss_i = exp(-log_var_i) * CE_i + log_var_i  (Kendall et al., 2018)\n\n        Label smoothing combats the overconfidence/poor calibration seen when\n        fine-tuning on this small a dataset (train loss -> 0 while val ECE rises).\n        diagnosis_class_weights (inverse-frequency) counters the severe class\n        imbalance in the 15-way diagnosis task (some classes have <30 cases).\n        diagnosis_loss_fn, if given, replaces weighted cross-entropy for the\n        diagnosis task only (e.g. a focal-loss ablation for rare classes).\n        """\n        ce = nn.functional.cross_entropy\n        per_task = {}\n        total = 0.0\n        for name, log_var in self.log_vars.items():\n            if name == "diagnosis" and diagnosis_loss_fn is not None:\n                task_loss = diagnosis_loss_fn(outputs[name], targets[name])\n            else:\n                weight = diagnosis_class_weights if name == "diagnosis" else None\n                task_loss = ce(outputs[name], targets[name], weight=weight, label_smoothing=label_smoothing)\n            weighted = torch.exp(-log_var) * task_loss + log_var\n            per_task[name] = task_loss.detach()\n            total = total + weighted\n        return total, per_task\n\n    def multitask_loss_clinical(\n        self,\n        outputs: dict,\n        targets: dict,\n        label_smoothing: float = 0.1,\n        diagnosis_class_weights: torch.Tensor | None = None,\n        diagnosis_loss_fn=None,\n    ) -> tuple[torch.Tensor, dict]:\n        """Ablation: fixed clinically-informed task weights (CLINICAL_TASK_WEIGHTS)\n        instead of the default learned uncertainty weighting -- tests whether\n        encoding the checklist\'s real point-values directly beats letting the\n        model learn task importance on its own.\n        """\n        ce = nn.functional.cross_entropy\n        per_task = {}\n        total = 0.0\n        total_weight = 0.0\n        for name in self.log_vars.keys():\n            weight = CLINICAL_TASK_WEIGHTS.get(name, 1.0)\n            if name == "diagnosis" and diagnosis_loss_fn is not None:\n                task_loss = diagnosis_loss_fn(outputs[name], targets[name])\n            else:\n                class_weight = diagnosis_class_weights if name == "diagnosis" else None\n                task_loss = ce(outputs[name], targets[name], weight=class_weight, label_smoothing=label_smoothing)\n            per_task[name] = task_loss.detach()\n            total = total + weight * task_loss\n            total_weight += weight\n        return total / total_weight, per_task\n\n\nif __name__ == "__main__":\n    import sys\n\n    backbone_name = sys.argv[1] if len(sys.argv) > 1 else "mobilenetv3_small_100"\n    model = MultiTaskDerm7pt(backbone_name, pretrained=False)\n    clinic = torch.randn(2, 3, 224, 224)\n    derm = torch.randn(2, 3, 224, 224)\n    meta = torch.randn(2, META_DIM)\n    out = model(clinic, derm, meta)\n    print(f"backbone={backbone_name}  params={sum(p.numel() for p in model.parameters()):,}")\n    for k, v in out.items():\n        print(f"  {k}: {tuple(v.shape)}")\n',
    "engine.py": '"""Training / evaluation loops, metrics (incl. calibration), for the\nDerm7pt multi-task model. Shared by local CPU smoke tests and the Kaggle\nGPU sweep so results are produced the same way everywhere.\n"""\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import f1_score\nfrom torchvision import transforms\n\nfrom labels import DIAGNOSIS_CLASSES, FIVE_WAY_CLASSES, FIVE_WAY_GROUPS\nfrom model import CRITERIA_NUM_CLASSES\n\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\nIMAGENET_STD = [0.229, 0.224, 0.225]\n\nTASK_NAMES = ["diagnosis", "malignant"] + list(CRITERIA_NUM_CLASSES.keys())\n\n# maps a 15-way diagnosis class index to its coarser 5-way group index (BCC/\n# NEV/MEL/MISC/SK), so predictions can be regrouped post-hoc for a direct\n# comparison against the original paper\'s 5-way diagnosis accuracy.\nFIVE_WAY_LOOKUP = np.array([FIVE_WAY_CLASSES.index(FIVE_WAY_GROUPS[c]) for c in DIAGNOSIS_CLASSES])\n\n\ndef build_transforms(img_size: int = 224, train: bool = True):\n    if train:\n        return transforms.Compose([\n            transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),\n            transforms.RandomHorizontalFlip(),\n            transforms.RandomRotation(20),\n            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),\n            transforms.ToTensor(),\n            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n        ])\n    return transforms.Compose([\n        transforms.Resize((img_size, img_size)),\n        transforms.ToTensor(),\n        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n    ])\n\n\ndef compute_diagnosis_class_weights(dataset) -> torch.Tensor:\n    """Inverse-frequency weights for the 15-way diagnosis task, computed from\n    a (possibly Subset-wrapped) Derm7ptDataset\'s train split -- counters the\n    severe imbalance (e.g. dermatofibroma/vascular lesion < 30 cases each).\n    """\n    from model import DIAGNOSIS_CLASSES\n\n    counts = torch.zeros(len(DIAGNOSIS_CLASSES))\n    underlying = dataset.dataset if hasattr(dataset, "dataset") else dataset\n    indices = dataset.indices if hasattr(dataset, "indices") else range(len(dataset))\n    for i in indices:\n        counts[underlying[i]["diagnosis"].item()] += 1\n    weights = 1.0 / counts.clamp(min=1)\n    weights = weights / weights.mean()\n    # cap at 10x the average weight so one near-empty class (e.g. a single-digit\n    # case count) can\'t dominate the gradient and destabilize training\n    weights = weights.clamp(max=10.0)\n    return weights\n\n\ndef _targets_from_batch(batch: dict) -> dict:\n    return {name: batch[name] for name in TASK_NAMES}\n\n\ndef _compute_loss(loss_model, outputs, targets, diagnosis_class_weights=None,\n                   loss_mode="uncertainty", diagnosis_loss_fn=None):\n    fn = loss_model.multitask_loss_clinical if loss_mode == "clinical" else loss_model.multitask_loss\n    return fn(outputs, targets, diagnosis_class_weights=diagnosis_class_weights,\n              diagnosis_loss_fn=diagnosis_loss_fn)\n\n\ndef train_one_epoch(model, loader, optimizer, device, raw_model=None, diagnosis_class_weights=None,\n                     loss_mode="uncertainty", diagnosis_loss_fn=None) -> float:\n    loss_model = raw_model if raw_model is not None else model\n    model.train()\n    total_loss = 0.0\n    n = 0\n    for batch in loader:\n        clinic = batch["clinic"].to(device)\n        derm = batch["derm"].to(device)\n        meta = batch["meta"].to(device)\n        targets = {k: v.to(device) for k, v in _targets_from_batch(batch).items()}\n\n        optimizer.zero_grad()\n        outputs = model(clinic, derm, meta)\n        loss, _ = _compute_loss(loss_model, outputs, targets, diagnosis_class_weights,\n                                 loss_mode, diagnosis_loss_fn)\n        loss.backward()\n        # the rare-class weighting can occasionally produce a large gradient spike\n        # (e.g. a batch containing a near-single-digit-count class) -- clip it so\n        # one such batch can\'t destabilize the whole training run\n        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)\n        optimizer.step()\n\n        bs = clinic.size(0)\n        total_loss += loss.item() * bs\n        n += bs\n    return total_loss / max(n, 1)\n\n\ndef focal_loss(logits: torch.Tensor, targets: torch.Tensor, weight: torch.Tensor | None = None,\n               gamma: float = 2.0, label_smoothing: float = 0.1) -> torch.Tensor:\n    """Focal loss (Lin et al., 2017) ablation for the rare diagnosis classes:\n    down-weights already-easy/confident examples (via (1-p_t)^gamma) so the\n    model keeps learning from the hard, often rare-class cases instead of\n    being dominated by gradient from the common ones, on top of the existing\n    inverse-frequency class weighting.\n    """\n    ce = F.cross_entropy(logits, targets, weight=weight, label_smoothing=label_smoothing, reduction="none")\n    pt = torch.exp(-ce)\n    return ((1 - pt) ** gamma * ce).mean()\n\n\ndef per_class_diagnosis_f1(probs: np.ndarray, labels: np.ndarray) -> dict:\n    """Per-class F1 for the 15-way diagnosis task -- macro-F1 alone hides\n    which specific rare classes (e.g. dermatofibroma, blue nevus) the model\n    actually fails on.\n    """\n    from sklearn.metrics import f1_score as _f1_score\n    preds = probs.argmax(axis=1)\n    per_class = _f1_score(labels, preds, average=None, labels=range(len(DIAGNOSIS_CLASSES)), zero_division=0)\n    return {cls: float(f1) for cls, f1 in zip(DIAGNOSIS_CLASSES, per_class)}\n\n\ndef expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:\n    """ECE for the top predicted class: how far confidence is from actual accuracy."""\n    confidences = probs.max(axis=1)\n    predictions = probs.argmax(axis=1)\n    correct = (predictions == labels).astype(np.float64)\n\n    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)\n    ece = 0.0\n    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):\n        mask = (confidences > lo) & (confidences <= hi)\n        if mask.sum() == 0:\n            continue\n        bin_acc = correct[mask].mean()\n        bin_conf = confidences[mask].mean()\n        ece += (mask.sum() / len(labels)) * abs(bin_acc - bin_conf)\n    return float(ece)\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device, raw_model=None) -> dict:\n    loss_model = raw_model if raw_model is not None else model\n    model.eval()\n    total_loss = 0.0\n    n = 0\n    all_logits = {name: [] for name in TASK_NAMES}\n    all_labels = {name: [] for name in TASK_NAMES}\n\n    for batch in loader:\n        clinic = batch["clinic"].to(device)\n        derm = batch["derm"].to(device)\n        meta = batch["meta"].to(device)\n        targets = {k: v.to(device) for k, v in _targets_from_batch(batch).items()}\n\n        outputs = model(clinic, derm, meta)\n        loss, _ = loss_model.multitask_loss(outputs, targets)\n        bs = clinic.size(0)\n        total_loss += loss.item() * bs\n        n += bs\n\n        for name in TASK_NAMES:\n            all_logits[name].append(outputs[name].cpu())\n            all_labels[name].append(targets[name].cpu())\n\n    metrics = {"val_loss": total_loss / max(n, 1)}\n    for name in TASK_NAMES:\n        logits = torch.cat(all_logits[name]).numpy()\n        labels = torch.cat(all_labels[name]).numpy()\n        probs = F.softmax(torch.from_numpy(logits), dim=1).numpy()\n        preds = probs.argmax(axis=1)\n\n        metrics[f"{name}_acc"] = float((preds == labels).mean())\n        metrics[f"{name}_macro_f1"] = float(\n            f1_score(labels, preds, average="macro", zero_division=0)\n        )\n        if name == "diagnosis":\n            metrics["diagnosis_ece"] = expected_calibration_error(probs, labels)\n            regrouped_preds = FIVE_WAY_LOOKUP[preds]\n            regrouped_labels = FIVE_WAY_LOOKUP[labels]\n            metrics["diagnosis_5way_acc"] = float((regrouped_preds == regrouped_labels).mean())\n            metrics["diagnosis_5way_macro_f1"] = float(\n                f1_score(regrouped_labels, regrouped_preds, average="macro", zero_division=0)\n            )\n\n    return metrics\n',
    "train_single_backbone.py": '"""Standalone single-backbone trainer for the Derm7pt multi-task sweep.\n\nRuns as its own OS process (invoked via subprocess.run from the Kaggle sweep\nnotebook) so a GPU/CUDA crash on one backbone (DataParallel + certain archs\nhas shown flaky cuDNN/misaligned-address failures on Kaggle\'s T4 pairs) can\nnever take down the rest of the sweep -- each backbone gets a fresh process\nand a fresh CUDA context.\n\nAlso bakes in the fixes for the accuracy/calibration problems seen in the\nfirst lightweight-tier run: label smoothing + diagnosis class weights (the\nmodel was overfitting fast -- train_loss -> ~0 while val ECE rose to 0.2-0.3\nand macro-F1 stayed far below accuracy because of severe class imbalance),\nstronger augmentation, a lower LR with cosine decay, and early stopping.\n"""\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n\nfrom dataset import Derm7ptDataset\nfrom engine import (\n    build_transforms,\n    compute_diagnosis_class_weights,\n    evaluate,\n    focal_loss,\n    per_class_diagnosis_f1,\n    train_one_epoch,\n)\nfrom model import MultiTaskDerm7pt\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--input-dir", required=True)\n    p.add_argument("--output-dir", required=True)\n    p.add_argument("--backbone", required=True)\n    p.add_argument("--tier", required=True)\n    p.add_argument("--epochs", type=int, default=40)\n    p.add_argument("--patience", type=int, default=12)\n    p.add_argument("--batch-size", type=int, default=32)\n    p.add_argument("--lr", type=float, default=2e-4)\n    p.add_argument("--img-size", type=int, default=224)\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--loss-mode", choices=["uncertainty", "clinical"], default="uncertainty",\n                   help="task-weighting ablation: learned uncertainty weighting (default) vs "\n                        "fixed clinical-checklist point-value weights")\n    p.add_argument("--diagnosis-loss", choices=["ce", "focal"], default="ce",\n                   help="diagnosis-task loss ablation: class-weighted cross-entropy (default) vs focal loss")\n    p.add_argument("--result-path", required=True)\n    p.add_argument("--limit-train", type=int, default=None, help="debug: cap train set size")\n    p.add_argument("--limit-valid", type=int, default=None, help="debug: cap valid set size")\n    p.add_argument("--no-pretrained", action="store_true", help="debug: skip downloading pretrained weights")\n    p.add_argument("--checkpoint-path", default=None,\n                   help="skip training entirely; load this existing checkpoint and only run the final valid+test evaluation")\n    return p.parse_args()\n\n\ndef fallback_row(args, error, dt=None):\n    return {\n        "tier": args.tier, "backbone": args.backbone, "params": None,\n        "train_time_s": round(dt, 1) if dt else None, "val_loss": None,\n        "diagnosis_acc": None, "diagnosis_macro_f1": None, "diagnosis_ece": None,\n        "malignant_acc": None, "malignant_macro_f1": None, "error": str(error),\n    }\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    try:\n        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        multi_gpu = torch.cuda.device_count() > 1\n\n        train_tf = build_transforms(args.img_size, train=True)\n        valid_tf = build_transforms(args.img_size, train=False)\n        train_ds = Derm7ptDataset(args.input_dir, "train", transform=train_tf)\n        valid_ds = Derm7ptDataset(args.input_dir, "valid", transform=valid_tf)\n        test_ds = Derm7ptDataset(args.input_dir, "test", transform=valid_tf)\n\n        if args.limit_train:\n            from torch.utils.data import Subset\n            train_ds = Subset(train_ds, range(min(args.limit_train, len(train_ds))))\n        if args.limit_valid:\n            from torch.utils.data import Subset\n            valid_ds = Subset(valid_ds, range(min(args.limit_valid, len(valid_ds))))\n\n        valid_loader = DataLoader(valid_ds, batch_size=args.batch_size, shuffle=False,\n                                   num_workers=args.num_workers)\n        test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False,\n                                  num_workers=args.num_workers)\n\n        os.makedirs(f"{args.output_dir}/checkpoints", exist_ok=True)\n\n        if args.checkpoint_path:\n            # eval-only mode: reuse an already-trained checkpoint (e.g. attached\n            # from a previous notebook\'s saved output) instead of retraining\n            raw_model = MultiTaskDerm7pt(args.backbone, pretrained=False, img_size=args.img_size).to(device)\n            raw_model.load_state_dict(torch.load(args.checkpoint_path, map_location=device))\n            model = nn.DataParallel(raw_model) if multi_gpu else raw_model\n            best_metrics = evaluate(model, valid_loader, device, raw_model=raw_model)\n            best_acc = best_metrics["diagnosis_acc"]\n            print(f"  loaded checkpoint {args.checkpoint_path}  val_diag_acc={best_acc:.4f}")\n        else:\n            diagnosis_class_weights = compute_diagnosis_class_weights(train_ds).to(device)\n            train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,\n                                       num_workers=args.num_workers)\n\n            diagnosis_loss_fn = None\n            if args.diagnosis_loss == "focal":\n                diagnosis_loss_fn = lambda logits, targets: focal_loss(\n                    logits, targets, weight=diagnosis_class_weights)\n\n            raw_model = MultiTaskDerm7pt(args.backbone, pretrained=not args.no_pretrained,\n                                          img_size=args.img_size).to(device)\n            model = nn.DataParallel(raw_model) if multi_gpu else raw_model\n            optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)\n            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)\n\n            best_acc, best_metrics, patience_ctr = -1.0, None, 0\n\n            for epoch in range(1, args.epochs + 1):\n                train_loss = train_one_epoch(model, train_loader, optimizer, device,\n                                              raw_model=raw_model,\n                                              diagnosis_class_weights=diagnosis_class_weights,\n                                              loss_mode=args.loss_mode,\n                                              diagnosis_loss_fn=diagnosis_loss_fn)\n                metrics = evaluate(model, valid_loader, device, raw_model=raw_model)\n                scheduler.step()\n\n                if metrics["diagnosis_acc"] > best_acc:\n                    best_acc = metrics["diagnosis_acc"]\n                    best_metrics = metrics\n                    patience_ctr = 0\n                    torch.save(raw_model.state_dict(),\n                               f"{args.output_dir}/checkpoints/{args.backbone}_best.pt")\n                else:\n                    patience_ctr += 1\n\n                if epoch % 5 == 0 or epoch == args.epochs or patience_ctr == 0:\n                    print(f"  epoch {epoch}/{args.epochs}  train_loss={train_loss:.4f}  "\n                          f"val_diag_acc={metrics[\'diagnosis_acc\']:.4f}  "\n                          f"val_diag_f1={metrics[\'diagnosis_macro_f1\']:.4f}  "\n                          f"ece={metrics[\'diagnosis_ece\']:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")\n\n                if patience_ctr >= args.patience:\n                    print(f"  early stopping at epoch {epoch} (no improvement for {args.patience} epochs)")\n                    break\n\n            # re-load the best checkpoint (not necessarily the last epoch) before\n            # the final test-set evaluation\n            raw_model.load_state_dict(torch.load(f"{args.output_dir}/checkpoints/{args.backbone}_best.pt",\n                                                  map_location=device))\n\n        # final test-set evaluation (395 held-out cases) -- the paper-comparable\n        # number; validation (used above for early stopping / model selection)\n        # is never touched during training, matching the original split roles.\n        test_metrics = evaluate(model, test_loader, device, raw_model=raw_model)\n        test_metrics = {f"test_{k}": v for k, v in test_metrics.items()}\n\n        # per-class diagnosis F1 on the test set -- macro-F1 alone hides which\n        # specific rare classes the model actually fails on\n        test_probs, test_labels = [], []\n        with torch.no_grad():\n            for batch in test_loader:\n                clinic = batch["clinic"].to(device)\n                derm = batch["derm"].to(device)\n                meta = batch["meta"].to(device)\n                out = model(clinic, derm, meta)\n                test_probs.append(torch.softmax(out["diagnosis"], dim=1).cpu())\n                test_labels.append(batch["diagnosis"])\n        test_probs = torch.cat(test_probs).numpy()\n        test_labels = torch.cat(test_labels).numpy()\n        per_class_f1 = per_class_diagnosis_f1(test_probs, test_labels)\n\n        dt = time.time() - t0\n        n_params = sum(p.numel() for p in raw_model.parameters())\n        row = {"tier": args.tier, "backbone": args.backbone, "params": n_params,\n               "loss_mode": args.loss_mode, "diagnosis_loss": args.diagnosis_loss,\n               "train_time_s": round(dt, 1), **best_metrics, **test_metrics,\n               "test_per_class_f1": per_class_f1}\n        with open(args.result_path, "w") as f:\n            json.dump(row, f)\n        print(f"  -> best val diagnosis_acc={best_acc:.4f}  "\n              f"test_diagnosis_acc={test_metrics[\'test_diagnosis_acc\']:.4f}  "\n              f"test_diagnosis_5way_acc={test_metrics[\'test_diagnosis_5way_acc\']:.4f}  "\n              f"({dt:.1f}s, {n_params:,} params)")\n\n    except Exception as e:\n        print(f"  !! {args.backbone} FAILED: {e}")\n        with open(args.result_path, "w") as f:\n            json.dump(fallback_row(args, e, time.time() - t0), f)\n        sys.exit(1)\n\n\nif __name__ == "__main__":\n    main()\n',
}
for _fname, _content in _MODULE_FILES.items():
    with open(os.path.join(OUTPUT_DIR, _fname), "w", encoding="utf-8") as _f:
        _f.write(_content)
print("wrote subprocess trainer files to", OUTPUT_DIR, ":", list(_MODULE_FILES.keys()))

wrote subprocess trainer files to /kaggle/working : ['labels.py', 'dataset.py', 'model.py', 'engine.py', 'train_single_backbone.py']


In [ ]:
# ---- run each loss-ablation config, each in its own subprocess ----
import subprocess
import sys
import json as _json

RESULTS_DIR = f'{OUTPUT_DIR}/ablation_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

results = []
for config_name, cfg in ABLATION_CONFIGS.items():
    print(f'\n========== CONFIG: {config_name} ({cfg}) ==========')
    result_path = f'{RESULTS_DIR}/{config_name}.json'
    cmd = [sys.executable, f'{OUTPUT_DIR}/train_single_backbone.py',
           '--input-dir', INPUT_DIR, '--output-dir', OUTPUT_DIR,
           '--backbone', BACKBONE, '--tier', 'lightweight',
           '--epochs', str(EPOCHS), '--patience', str(EARLY_STOP_PATIENCE),
           '--batch-size', str(BATCH_SIZE), '--lr', str(LR), '--img-size', str(IMG_SIZE),
           '--num-workers', str(NUM_WORKERS), '--result-path', result_path,
           '--loss-mode', cfg['loss_mode'], '--diagnosis-loss', cfg['diagnosis_loss']]
    for _attempt in range(2):
        proc = subprocess.run(cmd)
        if proc.returncode == 0 and os.path.exists(result_path):
            break
        note = ' -- retrying once in a fresh process' if _attempt == 0 else ' -- giving up after retry'
        print(f'  !! subprocess exited with code {proc.returncode} for {config_name}{note}')

    if os.path.exists(result_path):
        with open(result_path) as f:
            row = _json.load(f)
    else:
        row = {'config': config_name, 'error': 'no result file written'}
    row['config'] = config_name
    results.append(row)

========== CONFIG: baseline ({'loss_mode': 'uncertainty', 'diagnosis_loss': 'ce'}) ==========


  epoch 1/40  train_loss=11.0625  val_diag_acc=0.2365  val_diag_f1=0.1145  ece=0.1343  lr=2.00e-04
  epoch 2/40  train_loss=9.9383  val_diag_acc=0.3350  val_diag_f1=0.1814  ece=0.2074  lr=1.99e-04
  epoch 3/40  train_loss=9.2672  val_diag_acc=0.3399  val_diag_f1=0.2718  ece=0.1722  lr=1.97e-04
  epoch 4/40  train_loss=8.9053  val_diag_acc=0.4384  val_diag_f1=0.2840  ece=0.2189  lr=1.95e-04
  epoch 5/40  train_loss=8.5766  val_diag_acc=0.3990  val_diag_f1=0.2635  ece=0.1539  lr=1.92e-04
  epoch 9/40  train_loss=7.1992  val_diag_acc=0.4532  val_diag_f1=0.3146  ece=0.1523  lr=1.76e-04
  epoch 10/40  train_loss=7.1733  val_diag_acc=0.4384  val_diag_f1=0.2996  ece=0.1265  lr=1.71e-04
  epoch 11/40  train_loss=6.7555  val_diag_acc=0.5172  val_diag_f1=0.3659  ece=0.1714  lr=1.65e-04
  epoch 15/40  train_loss=5.6220  val_diag_acc=0.4926  val_diag_f1=0.3736  ece=0.1305  lr=1.38e-04
  epoch 16/40  tra

In [ ]:
# ---- comparison: overall metrics + per-class F1 for the rarest classes ----
df = pd.DataFrame(results)
df.to_csv(f'{OUTPUT_DIR}/ablation_results.csv', index=False)

print("=== Overall metrics ===")
cols = ['config', 'diagnosis_acc', 'diagnosis_macro_f1', 'test_diagnosis_acc',
        'test_diagnosis_macro_f1', 'test_diagnosis_5way_acc', 'test_diagnosis_5way_macro_f1']
print(df[[c for c in cols if c in df.columns]].to_string(index=False))

print("\n=== Per-class test F1 (rarest classes -- where class-imbalance ablations should help most) ===")
RARE_CLASSES = ['dermatofibroma', 'blue nevus', 'vascular lesion', 'melanosis', 'lentigo']
for _, row in df.iterrows():
    pcf1 = row.get('test_per_class_f1', {})
    if isinstance(pcf1, dict) and pcf1:
        vals = ', '.join(f"{c}={pcf1.get(c, float('nan')):.3f}" for c in RARE_CLASSES if c in pcf1)
        print(f"  {row['config']:10s}  {vals}")

print("\nbaseline (previous 512px+meta run, Nov 2026): test_diagnosis_5way_acc = 0.6734, "
      "test_diagnosis_macro_f1 = 0.3617")

=== Overall metrics ===
  config  diagnosis_acc  diagnosis_macro_f1  test_diagnosis_acc  test_diagnosis_macro_f1  test_diagnosis_5way_acc  test_diagnosis_5way_macro_f1
baseline       0.615764            0.463025            0.602532                 0.424956                 0.693671                      0.523689
clinical       0.576355            0.397850            0.554430                 0.346333                 0.668354                      0.481478
   focal       0.458128            0.361048            0.382278                 0.300829                 0.617722                      0.447853

=== Per-class test F1 (rarest classes -- where class-imbalance ablations should help most) ===
  baseline    dermatofibroma=0.378, blue nevus=0.552, vascular lesion=0.571, melanosis=0.909, lentigo=0.636
  clinical    dermatofibroma=0.359, blue nevus=0.640, vascular lesion=0.381, melanosis=0.769, lentigo=0.421
  focal       dermatofibroma=0.255, blue nevus=0.412, vascular lesion=0.522, melanosis=0